# Cardiovascular Disease Prediction
## Notebook 03: Support Vector Machine (SVM)
---
**Author:** MS26912370 (Member 2)
**Algorithm:** Support Vector Machine with RBF Kernel

### Algorithm Background

A **Support Vector Machine (SVM)** is a supervised learning algorithm that finds the
optimal decision boundary (**hyperplane**) that maximises the **margin** between classes.

#### Key Concepts:
- **Maximum Margin Classifier:** The hyperplane is positioned to maximise the distance
  to the nearest training points (the **support vectors**).
- **Kernel Trick:** For non-linearly separable data, SVM maps features into a
  higher-dimensional space using a **kernel function**.
  - **RBF (Radial Basis Function):** `K(x,z) = exp(-γ||x-z||²)` — maps to infinite dimensions.
- **Regularisation (C):** Controls the trade-off between maximising margin and minimising
  training errors. Low C = wider margin (more regularisation); High C = narrower margin.
- **Gamma (γ):** Defines the influence radius of a single training example.
  Low γ = large radius (smooth boundary); High γ = small radius (complex boundary).

#### Why SVM for this Dataset?
1. Effective in **high-dimensional spaces** even with many features
2. **Memory efficient** — uses only support vectors for the decision function
3. **Kernel trick** captures non-linear patterns in cardiovascular risk factors
4. Provides **probability calibration** via Platt scaling (`probability=True`)
5. Contrasts well with Random Forest for academic comparison

#### SVM vs Random Forest — Key Differences
| Aspect              | Random Forest          | SVM                        |
|--------------------|------------------------|----------------------------|
| Approach           | Ensemble of trees      | Single optimal hyperplane  |
| Feature scaling    | Not required           | **Required** (done in NB01)|
| Interpretability   | Feature importance     | Support vectors            |
| Training speed     | Fast                   | Slower for large N         |
| Kernel trick       | No                     | Yes (RBF)                  |

In [ ]:
# Import all required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os
import warnings

from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve, ConfusionMatrixDisplay
)

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
os.makedirs('plots', exist_ok=True)

print("✓ Libraries imported successfully.")

In [ ]:
import os

# ============================================================
# ENVIRONMENT SETUP — comment out the block you are NOT using
# ============================================================

# ── OPTION A: Google Colab ───────────────────────────────────
# Mounts Drive and changes to your project folder so
# preprocessed_data.pkl (saved by NB01) can be found here.
from google.colab import drive
drive.mount('/content/drive')
DRIVE_PATH = '/content/drive/MyDrive/ML-Assignment/attemp2'
os.chdir(DRIVE_PATH)
print(f"✓ Working directory: {DRIVE_PATH}")

# ── OPTION B: Local Jupyter ──────────────────────────────────
# Comment out Option A above and uncomment the line below.
# All pkl files must be in the same folder as this notebook.
# LOCAL_PATH = os.path.dirname(os.path.abspath("__file__"))
# os.chdir(LOCAL_PATH)


In [ ]:
# Load preprocessed data (created by Notebook 01)
with open('preprocessed_data.pkl', 'rb') as f:
    data = pickle.load(f)

X_train      = data['X_train']
X_test       = data['X_test']
y_train      = data['y_train']
y_test       = data['y_test']
feature_names = data['feature_names']

print(f"✓ Preprocessed data loaded.")
print(f"  Train: {X_train.shape[0]:,} samples | Test: {X_test.shape[0]:,} samples")
print(f"  Features ({X_train.shape[1]}): {feature_names}")
print("\nNote: Data is already StandardScaler-normalised — ready for SVM.")

## 1. Baseline SVM Model

First, train an SVM with default RBF kernel parameters to get a baseline.
Because the full dataset (~56,000 training samples) is large for SVM, we
use a **stratified subsample of 15,000** for the initial baseline check.

In [ ]:
# Stratified subsample for baseline speed test
from sklearn.utils import resample

indices = np.arange(len(X_train))
sample_size = min(15000, len(X_train))
np.random.seed(42)

# Stratified sample
X_sample, y_sample = resample(
    X_train, y_train,
    n_samples=sample_size,
    random_state=42,
    stratify=y_train
)
print(f"Using {sample_size:,} stratified samples for baseline SVM.")
print(f"Class balance in sample: {np.bincount(y_sample) / sample_size}")

In [ ]:
# Baseline SVM – RBF kernel, default C=1, gamma='scale'
svm_baseline = SVC(kernel='rbf', probability=True, random_state=42)
svm_baseline.fit(X_sample, y_sample)

y_pred_base_svm = svm_baseline.predict(X_test)
y_prob_base_svm = svm_baseline.predict_proba(X_test)[:, 1]

# Baseline metrics
print("=== Baseline SVM Performance ===")
print(f"  Accuracy  : {accuracy_score(y_test, y_pred_base_svm):.4f}")
print(f"  Precision : {precision_score(y_test, y_pred_base_svm):.4f}")
print(f"  Recall    : {recall_score(y_test, y_pred_base_svm):.4f}")
print(f"  F1-Score  : {f1_score(y_test, y_pred_base_svm):.4f}")
print(f"  ROC-AUC   : {roc_auc_score(y_test, y_prob_base_svm):.4f}")
print(f"  Support vectors: {svm_baseline.n_support_.sum():,}")

## 2. Hyperparameter Tuning with GridSearchCV

We tune `C` (regularisation) and `gamma` (kernel width) using **5-Fold Stratified CV**,
optimising for **F1-score**.

| Parameter | Values          | Effect                                          |
|-----------|----------------|-------------------------------------------------|
| `C`       | 0.1, 1, 10, 100 | Low=underfitting, High=overfitting              |
| `gamma`   | 'scale', 0.1, 0.01 | Low=smooth boundary, High=complex boundary |

> **Note:** GridSearchCV is run on a **20,000-sample stratified subset** to manage
> computational cost. The best model is then trained on the **full training set**.